In [0]:
%sql
-- ============================================================================
-- SQL DATABASE SETUP: GoHighLevel CRM Contacts for Data Engineering Practice
-- ============================================================================
-- Purpose: To Create a realistic CRM database in a Data 'Lake' (vs Data Warehouse vs Data Lakehouse)  
-- with 10000 dummy records to practice essential SQL clauses & Data Cleaning and prepare to answer real business questions.
--
-- Context: This mirrors a real-world small business start-up using Azure/Databricks architecture where GoHighLevel data
--          lands in the 'raw' schema/db, ready for transformation in dbt.
-- ============================================================================


-- STEP 1: CREATE DATABASE
-- ============================================================================
-- Use Microsoft Azure and Databricks to create a CATALOG (in Databricks namespace for Database) & SCHEMA for my data with Notebooks and SQL.
-- 


-- STEP 2: CREATE SCHEMA (i.e in Databricks namespace for tables)
-- ============================================================================
-- The 'raw' schema is where unprocessed GoHighLevel exports land.
-- i'll also create additional practice 'stg' (staging) and 'mart' schemas in dbt.
USE CATALOG `bh2026-winford-uc-dev`;
CREATE SCHEMA IF NOT EXISTS raw;
USE SCHEMA raw;

SELECT current_catalog(), current_schema();

In [0]:
%sql
-- DROP raw.crm_contacts table for re-run
DROP TABLE IF EXISTS raw.crm_contacts

In [0]:
%sql
-- STEP 3: CREATE raw.crm_contacts TABLE
-- ============================================================================
-- This table mirrors GoHighLevel's contact export structure.
-- Fields: ID, name, email, phone, status, region, created_date, deal_value, source
--
-- NOTE: No constraints enforced here (no NOT NULL, no PRIMARY KEY).
-- In a data lakehouse, the RAW layer accepts everything as-is.
-- Data quality checks belong in the staging/transformation layer (e.g., dbt tests).

CREATE TABLE raw.crm_contacts (
    contact_id INT,
    name STRING,
    email STRING,
    phone STRING,
    status STRING,
    region STRING,
    created_date DATE,
    deal_value DECIMAL(10, 2),
    source STRING,
    created_at TIMESTAMP
);

In [0]:
%sql
-- STEP 4: INSERT 10,000 DELIBERATELY MESSY RECORDS
-- ============================================================================
-- This data has INTENTIONAL quality issues for practicing data cleansing:
--
-- | Quality Check                        | How It's Represented                          |
-- |--------------------------------------|-----------------------------------------------|
-- | Null/Missing Values                  | NULLs & empty strings across all columns      |
-- | Primary Key Uniqueness               | Duplicate contact_ids (rows 7001-8000 = 1-1000)|
-- | Duplicate Record Detection           | Exact duplicate batch (rows 9001-9500 = 1-500)|
-- | Referential Integrity                | Invalid regions (New York, Paris, Tokyo)      |
-- | Data Type Validation                 | Phone has letters ('N/A','UNKNOWN','TBC')     |
-- | Numeric Range Validation             | Negative deal_values, values > £500k          |
-- | String Length & Pattern              | Invalid emails, short phones, excess whitespace|
-- | Allowed Value / Domain               | Invalid statuses ('active','closed','won')    |
-- | Business Rule Consistency            | Converted leads with £0 deals, disqualified w/ high deals |
-- | Cross-Column Consistency             | created_at BEFORE created_date                |
-- | Timeliness / Freshness               | Dates from 10 years ago, future dates         |
-- | Completeness Check                   | Rows with ALL optional fields NULL            |
-- | Volume Check                         | Compare vs expected 10k (duplicates push >10k)|
-- | Statistical Distribution             | Region skew (70% London in some batches)      |
-- | Outlier Detection                    | Deal values £999,999 and £0.01                |
-- | Schema Drift Detection               | Mixed casing, extra spaces in categoricals    |
-- | Duplicate File Ingestion             | Batch of rows 1-500 repeated exactly          |
-- | Negative / Invalid Values            | Negative deal_value, negative contact_id      |
-- | Percentage / Total Consistency       | deal_values that don't sum to expected totals |
-- | Hierarchy Validation                 | Regions not in valid UK hierarchy             |
-- | Audit Column Consistency             | created_at = NULL, or before created_date     |
-- ============================================================================

INSERT INTO raw.crm_contacts
(contact_id, name, email, phone, status, region, created_date, deal_value, source, created_at)

WITH base AS (
    SELECT EXPLODE(SEQUENCE(1, 10000)) AS rn
)
SELECT
    -- CONTACT_ID: duplicates (7001-8000 repeat 1-1000), negatives, NULLs
    CASE
        WHEN rn BETWEEN 7001 AND 8000 THEN rn - 7000
        WHEN rn BETWEEN 9001 AND 9500 THEN rn - 9000
        WHEN rn % 500 = 0 THEN NULL
        WHEN rn % 333 = 0 THEN -1 * rn
        ELSE rn
    END AS contact_id,

    -- NAME: NULLs, empty strings, whitespace, inconsistent casing, special chars
    CASE
        WHEN rn % 100 = 0 THEN NULL
        WHEN rn % 73 = 0 THEN ''
        WHEN rn % 50 = 0 THEN CONCAT('  Contact_', rn, '  ')
        WHEN rn % 37 = 0 THEN CONCAT('CONTACT_', rn)
        WHEN rn % 29 = 0 THEN CONCAT('contact_', rn)
        WHEN rn % 23 = 0 THEN CONCAT('Contact #', rn, ' (dup?)')
        WHEN rn BETWEEN 9001 AND 9500 THEN CONCAT('Contact_', rn - 9000)
        ELSE CONCAT('Contact_', rn)
    END AS name,

    -- EMAIL: NULLs, invalid formats, duplicates, whitespace, missing domain
    CASE
        WHEN rn % 15 = 0 THEN NULL
        WHEN rn % 41 = 0 THEN 'not-an-email'
        WHEN rn % 53 = 0 THEN CONCAT('contact_', rn)
        WHEN rn % 67 = 0 THEN CONCAT('CONTACT_', rn, '@EXAMPLE.COM')
        WHEN rn % 83 = 0 THEN CONCAT(' contact_', rn, '@example.com ')
        WHEN rn % 97 = 0 THEN 'test@@double-at.com'
        WHEN rn % 109 = 0 THEN ''
        WHEN rn % 127 = 0 THEN 'plaintext'
        WHEN rn BETWEEN 7001 AND 7500 THEN CONCAT('contact_', rn - 7000, '@example.com')
        WHEN rn BETWEEN 9001 AND 9500 THEN CONCAT('contact_', rn - 9000, '@example.com')
        ELSE CONCAT('contact_', rn, '@example.com')
    END AS email,

    -- PHONE: NULLs, non-numeric, too short/long, all zeros, placeholder text
    CASE
        WHEN rn % 12 = 0 THEN NULL
        WHEN rn % 47 = 0 THEN '0000000000'
        WHEN rn % 59 = 0 THEN 'N/A'
        WHEN rn % 71 = 0 THEN '+44123'
        WHEN rn % 89 = 0 THEN 'UNKNOWN'
        WHEN rn % 97 = 0 THEN 'TBC'
        WHEN rn % 103 = 0 THEN ''
        WHEN rn % 157 = 0 THEN '+44 7911 123456'
        WHEN rn % 173 = 0 THEN '07911123456'
        ELSE CONCAT('+44', LPAD(CAST(CAST(RAND() * 9000000000 + 1000000000 AS BIGINT) AS STRING), 10, '0'))
    END AS phone,

    -- STATUS: NULLs, typos, invalid values, inconsistent casing
    CASE
        WHEN rn % 200 = 0 THEN NULL
        WHEN rn % 61 = 0 THEN 'Converted'
        WHEN rn % 43 = 0 THEN 'PENDING'
        WHEN rn % 79 = 0 THEN 'convertd'
        WHEN rn % 91 = 0 THEN 'active'
        WHEN rn % 103 = 0 THEN 'closed'
        WHEN rn % 107 = 0 THEN 'disqualifed'
        WHEN rn % 113 = 0 THEN 'won'
        WHEN rn % 139 = 0 THEN ''
        WHEN rn % 151 = 0 THEN ' pending '
        WHEN RAND() < 0.4 THEN 'converted'
        WHEN RAND() < 0.5 THEN 'pending'
        ELSE 'disqualified'
    END AS status,

    -- REGION: NULLs, invalid, typos, non-UK, inconsistent casing, whitespace
    CASE
        WHEN rn % 150 = 0 THEN NULL
        WHEN rn % 57 = 0 THEN 'london'
        WHEN rn % 63 = 0 THEN 'MANCHESTER'
        WHEN rn % 77 = 0 THEN 'Birmigham'
        WHEN rn % 87 = 0 THEN 'New York'
        WHEN rn % 93 = 0 THEN 'Paris'
        WHEN rn % 101 = 0 THEN ' Leeds '
        WHEN rn % 117 = 0 THEN 'Tokyo'
        WHEN rn % 131 = 0 THEN 'Edinburg'
        WHEN rn % 143 = 0 THEN ''
        WHEN rn % 167 = 0 THEN 'Manchester '
        WHEN rn BETWEEN 8501 AND 8850 THEN 'London'
        WHEN RAND() < 0.3 THEN 'London'
        WHEN RAND() < 0.4 THEN 'Manchester'
        WHEN RAND() < 0.5 THEN 'Birmingham'
        WHEN RAND() < 0.6 THEN 'Leeds'
        ELSE 'Edinburgh'
    END AS region,

    -- CREATED_DATE: NULLs, future dates, very old dates, epoch, impossible dates
    CASE
        WHEN rn % 180 = 0 THEN NULL
        WHEN rn % 111 = 0 THEN DATE_ADD(CURRENT_DATE, CAST(RAND() * 365 + 1 AS INT))
        WHEN rn % 131 = 0 THEN DATE_SUB(CURRENT_DATE, CAST(RAND() * 3650 AS INT))
        WHEN rn % 151 = 0 THEN CAST('1900-01-01' AS DATE)
        WHEN rn % 171 = 0 THEN CAST('2099-12-31' AS DATE)
        WHEN rn % 191 = 0 THEN CAST('1970-01-01' AS DATE)
        ELSE DATE_SUB(CURRENT_DATE, CAST(RAND() * 30 AS INT))
    END AS created_date,

    -- DEAL_VALUE: NULLs, negatives, extreme outliers, zeros, suspicious patterns
    CASE
        WHEN rn % 10 = 0 THEN NULL
        WHEN rn % 113 = 0 THEN -1 * ROUND(RAND() * 5000, 2)
        WHEN rn % 127 = 0 THEN 999999.99
        WHEN rn % 137 = 0 THEN 0.00
        WHEN rn % 149 = 0 THEN 0.01
        WHEN rn % 163 = 0 THEN -99999.99
        WHEN rn BETWEEN 8001 AND 8200 THEN ROUND(RAND() * 500000 + 100000, 2)
        WHEN rn % 181 = 0 THEN 12345.00
        WHEN rn % 197 = 0 THEN 50000.01
        ELSE ROUND(RAND() * 50000, 2)
    END AS deal_value,

    -- SOURCE: NULLs, invalid values, inconsistent casing, empty, unknown
    CASE
        WHEN rn % 8 = 0 THEN NULL
        WHEN rn % 69 = 0 THEN 'Web'
        WHEN rn % 81 = 0 THEN 'REFERRAL'
        WHEN rn % 99 = 0 THEN 'social_media'
        WHEN rn % 109 = 0 THEN 'unknown'
        WHEN rn % 119 = 0 THEN ''
        WHEN rn % 129 = 0 THEN 'Facebook'
        WHEN rn % 141 = 0 THEN 'google_ads'
        WHEN rn % 153 = 0 THEN 'cold_call'
        WHEN RAND() < 0.6 THEN 'web'
        ELSE 'referral'
    END AS source,

    -- CREATED_AT: NULLs, far future, epoch, BEFORE created_date (cross-column inconsistency)
    CASE
        WHEN rn % 5 = 0 THEN NULL
        WHEN rn % 121 = 0 THEN CAST('2099-12-31 23:59:59' AS TIMESTAMP)
        WHEN rn % 139 = 0 THEN CAST('1970-01-01 00:00:00' AS TIMESTAMP)
        WHEN rn % 157 = 0 THEN CAST(DATE_SUB(CURRENT_DATE, 365) AS TIMESTAMP)
        WHEN rn % 179 = 0 THEN CAST(DATE_SUB(CURRENT_DATE, 730) AS TIMESTAMP)
        ELSE CURRENT_TIMESTAMP
    END AS created_at

FROM base;

In [0]:
%sql

-- STEP 5: VERIFY THE DATA
-- ============================================================================
-- Run these queries to confirm your setup is correct.

-- Check row count
-- SELECT COUNT(*) AS total_number_of_records FROM crm_contacts;
-- SELECT COUNT(*) AS total_number_of_records_in_db FROM raw.crm_contacts;

-- SELECT COUNT(*) AS total_number_of_records_today FROM raw.crm_contacts;

-- Preview first 10 records (uncomment below and run separately)
-- SELECT * FROM raw.crm_contacts ORDER BY contact_id LIMIT 10;
-- SELECT * FROM crm_contacts WHERE contact_id IS NULL ;

-- SELECT region FROM raw.crm_contacts WHERE region = 'Birmingham' LIMIT 50;
-- SELECT region from raw.crm_contacts WHERE region = 'Manchester' limit 20;

-- Check data distribution by region (uncomment and run separately)
-- SELECT region, COUNT(*) AS contact_count FROM crm_contacts GROUP BY region ORDER BY contact_count DESC;

-- Check data distribution by status (uncomment and run separately)
-- SELECT status, COUNT(*) AS contact_count FROM raw.crm_contacts GROUP BY status ORDER BY contact_count DESC;
-- SELECT region, COUNT(*) AS contact_countwd FROM crm_contacts GROUP BY region ORDER BY contact_countwd DESC;

-- Check conversion rate (uncomment and run separately)
-- SELECT 
--     CAST(SUM(CASE WHEN status = 'converted' THEN 1 ELSE 0 END) AS FLOAT) 
--     / COUNT(*) * 100 AS conversion_rate_pct
-- FROM raw.crm_contacts;

In [0]:
%sql

-- ============================================================================
-- NOW YOU'RE READY TO PRACTICE!
-- ============================================================================
-- 
-- Run the business question query from the learning reference:
--
-- "Find the top 5 UK regions by number of converted leads this month"
--
-- Copy this query and run it against your new database:
--

WITH this_month_contacts AS (
  SELECT contact_id, name, region, status, created_date
  FROM raw.crm_contacts
  WHERE created_date >= DATE_TRUNC('month', CURRENT_DATE)
    AND region IN ('London', 'Manchester', 'Birmingham', 'Leeds', 'Edinburgh')
)
SELECT
  region,
  COUNT(*) AS converted_leads
FROM this_month_contacts
WHERE status = 'converted'
GROUP BY region
HAVING COUNT(*) > 0
ORDER BY converted_leads DESC
LIMIT 5;

-- Expected output: Top 5 regions with their converted lead counts
--
-- ============================================================================


-- BONUS: PRACTICE QUERIES
-- ============================================================================

-- 1. Find all pending leads in London
-- SELECT * FROM raw.crm_contacts 
-- WHERE region = 'London' AND status = 'pending';

-- 2. Average deal value by region
-- SELECT region, AVG(deal_value) AS avg_deal_value 
-- FROM raw.crm_contacts 
-- WHERE deal_value > 0 
-- GROUP BY region 
-- ORDER BY avg_deal_value DESC;

-- 3. Leads created in the last 7 days
-- SELECT * FROM raw.crm_contacts 
-- WHERE created_date >= DATEADD(DAY, -7, CAST(GETDATE() AS DATE)) 
-- ORDER BY created_date DESC;

-- 4. High-value deals (>£10,000) by source
-- SELECT source, COUNT(*) AS high_value_deals, SUM(deal_value) AS total_value
-- FROM raw.crm_contacts
-- WHERE deal_value > 10000
-- GROUP BY source
-- ORDER BY total_value DESC;

-- 5. Conversion rate by region
-- SELECT 
--     region,
--     COUNT(*) AS total_leads,
--     SUM(CASE WHEN status = 'converted' THEN 1 ELSE 0 END) AS converted,
--     ROUND(
--         CAST(SUM(CASE WHEN status = 'converted' THEN 1 ELSE 0 END) AS FLOAT) 
--         / COUNT(*) * 100, 
--         2
--     ) AS conversion_rate_pct
-- FROM raw.crm_contacts
-- GROUP BY region
-- ORDER BY conversion_rate_pct DESC;

In [0]:
%sql
